### Use LangChain's built-in tools/ creating custom tools and understand how tools are called

In [1]:
import langchain
import langchain_core
import langchain_openai
import langchain_community
from importlib.metadata import version
import transformers
import numpy as np

print(version("langchain"))
print(version("langchain-core"))
print(version("langchain-openai"))
print(version("langchain-community"))

print("langchain_v",langchain.__version__)
print("langchain_core",langchain_core.__version__)
print("langchain_community",langchain_community.__version__)
print("transformers",transformers.__version__)
print("numpy",np.__version__)

c:\study\AI\IITM_Agentic_AI_Training\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1.0.1
1.0.1
1.0.1
0.4
langchain_v 1.0.1
langchain_core 1.0.1
langchain_community 0.4
transformers 4.57.6
numpy 1.26.4


In [2]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [28]:
#!pip install wikipedia

In [29]:
#!pip list | FINDSTR wikipedia

In [4]:
# ------------------------------------------------------------
# Step 1: Initialize Wikipedia tool
# ------------------------------------------------------------
#WikipediaQueryRun already inherits from BaseTool
wikipedia_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper()
)

In [5]:
result = wikipedia_tool.invoke("Albert Einstein")
#print(result)

In [6]:
from langchain_core.tools import Tool

calculator_tool = Tool(
    name="Calculator",
    func=lambda x: str(eval(x)),
    description="Useful for running math expressions like '3+12'"
)

from math import sqrt 
#Using pre-defined function
def calc(expression: str) -> str:
    return str(eval(expression))

calculator_tool2 = Tool(
    name="Calculator",
    func=calc,
    description="Evaluate math expressions"
)

In [7]:
result = calculator_tool.invoke("3 + 12")
print(result)

15


In [8]:
#Or LangChain tools also support .run()
result = calculator_tool.run("10 * 25")
print(result)

250


In [9]:
result = calculator_tool2.invoke("sqrt(16) + 5")
print(result)

9.0


In [10]:
#Other option i.e. creating a custom tool
from langchain.tools import tool
from langchain_community.utilities import WikipediaAPIWrapper

wiki_api = WikipediaAPIWrapper()

@tool
def wikipedia_search(query: str) -> str:
    """Useful for fetching facts from Wikipedia."""
    return wiki_api.run(query)
#uncomment below to test
#wikipedia_search.invoke("Marie Curie")

@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    return str(eval(expression))
#uncomment below to test
#calculator.invoke("3 + 4 * 2")

from langchain.tools import tool
import math

@tool
def calculator2(expression: str) -> str:
    """Evaluate mathematical expressions safely."""
    allowed_names = {
        "abs": abs,
        "round": round,
        "sqrt": math.sqrt,
        "pow": pow,
        "pi": math.pi,
        "e": math.e,
    }
    return str(eval(expression, {"__builtins__": {}}, allowed_names))
#uncomment below to test   
#calculator2.invoke("3 + 4 * 2")

In [70]:
#Or
'''
print("=== Wikipedia Tool ===")
print(wikipedia_search.invoke("Marie Curie"))

print("\n=== Calculator Tool ===")
print(calculator.invoke("3 + 4 * 2"))

print("\n=== Safe Calculator Tool ===")
print(calculator2.invoke("sqrt(81) + pi"))
'''

'\nprint("=== Wikipedia Tool ===")\nprint(wikipedia_search.invoke("Marie Curie"))\n\nprint("\n=== Calculator Tool ===")\nprint(calculator.invoke("3 + 4 * 2"))\n\nprint("\n=== Safe Calculator Tool ===")\nprint(calculator2.invoke("sqrt(81) + pi"))\n'

### Build a simple agent using a prompt template + 2 tools (e.g., calculator + search)

In [11]:
from langchain_huggingface import HuggingFacePipeline
from langchain_community.agent_toolkits.load_tools import load_tools as lc_load_tools
from transformers import pipeline
from langchain_core.runnables import RunnableSequence

In [12]:
hf_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256,
)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

c:\study\AI\IITM_Agentic_AI_Training\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Ankur\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Device set to use cpu


In [14]:
#To use existing (built-in)tools
#!pip install numexpr
tools = lc_load_tools(["wikipedia", "llm-math"], llm=llm)

#we can also do to add our custom tools
#tools = lc_load_tools(["wikipedia", "llm-math"], llm=llm) + [calculator2]

In [15]:
type(tools)

list

In [16]:
for t in tools:
    print(t.name, "->", t.description)

wikipedia -> A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.
Calculator -> Useful for when you need to answer questions about math.


In [17]:
#Building simple agent
def get_tool_by_keyword(keyword):
    for t in tools:
        if keyword.lower() in t.name.lower():
            return t
    return None

def agent(query: str):
    q = query.lower()

    # Math routing
    if any(x in q for x in ["*", "+", "-", "/", "calculate", "math"]):
        tool = get_tool_by_keyword("math") or get_tool_by_keyword("calc")
        if tool:
            return tool.run(query)

    # Wikipedia routing
    if any(x in q for x in ["who", "what", "where", "capital", "wiki"]):
        tool = get_tool_by_keyword("wikipedia")
        if tool:
            return tool.run(query)

    # Fallback
    return llm.invoke(query)

In [18]:
#agent("What is the capital of France?")
agent("Explain quantum computing")

'Quantum computing (QCC) is a technique that combines the properties of a quantum wave with the properties of a non-resonant wave. Quantum computing is a technique that combines the properties of a quantum wave with the properties of a nonresonant wave. Quantum computing is a technique that combines the properties of a quantum wave with the properties of a nonresonant wave. Quantum computing is a technique that combines the properties of a quantum wave with the properties of a nonresonant wave. Quantum computing is a technique that combines the properties of a quantum wave with the properties of a nonresonant wave. Quantum computing is a technique that combines the properties of a quantum wave with the properties of a nonresonant wave. Quantum computing is a technique that combines the properties of a quantum wave with the properties of a nonresonant wave. Quantum computing is a technique that combines the properties of a quantum wave with the properties of a nonresonant wave. Quantum 

In [19]:
#Using tools, llm and templates
from langchain_core.prompts import PromptTemplate,ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = "Please write a {length} review of the book {book_title}."
prompt = PromptTemplate(
    input_variables=["length", "book_title"],
    template=template
)

llm_chain = prompt | llm | StrOutputParser()

In [20]:
#Approach 1: Tool-first, then LLM
#Use tools when the query clearly needs factual information. Then pass the gathered facts to the LLM for final writing.
'''
This is ideal when:

factual accuracy matters
you need external information
the LLM should synthesize rather than invent
'''
def review_with_tools_first(book_title: str, length: str = "short"):
    # Fetch facts from Wikipedia
    wiki_info = wikipedia_search.invoke(book_title)

    enhanced_prompt = f"""
    Using the following factual information:

    {wiki_info}

    Write a {length} review of the book {book_title}.
    Include a brief summary, themes, and overall impression.
    """

    return llm.invoke(enhanced_prompt)

In [21]:
print(review_with_tools_first("The Hobbit", "short"))

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [22]:
#Approach 2: LLM-first, then Tools if Needed
#Let the LLM answer first. If the response is weak, incomplete, or uncertain, then call tools.
'''
This is ideal when:

many questions can be answered directly
you want to minimize tool usage
tools are expensive or slow
'''
from langchain_core.tools import tool
import wikipedia

@tool
def wikipedia_search(query: str) -> str:
    """Fetch a short summary from Wikipedia for a given topic."""
    try:
        return wikipedia.summary(query, sentences=5)
    except Exception as e:
        return f"Wiki lookup failed: {e}"

def review_with_llm_first(book_title: str, length: str = "short"):
    initial_review = llm_chain.invoke({
        "length": length,
        "book_title": book_title
    })

    needs_enrichment = len(initial_review.split()) < 30

    if needs_enrichment:
        wiki_info = wikipedia_search.invoke(book_title)

        enhanced_prompt = f"""
        Improve this review:

        Review:
        {initial_review}

        Facts:
        {wiki_info}

        Write a {length} review of {book_title}.
        """

        return llm.invoke(enhanced_prompt)

    return initial_review

In [23]:
print(review_with_llm_first("The Hobbit", "short"))

The Hobbit is a well-written, well-acted, and well-written fantasy, but it's also a bit too sentimental for its own good.


In [24]:
#Router pattern
def book_review_agent(book_title: str, length: str = "short", use_tools=True):
    if use_tools:
        return review_with_tools_first(book_title, length)
    else:
        return review_with_llm_first(book_title, length)

In [25]:
print(book_review_agent("The Hobbit", "short", use_tools=True))

The Hobbit is set in Middle-earth and follows home-loving Bilbo Baggins, the titular hobbit who joins the wizard Gandalf and the thirteen dwarves of Thorin's Company on a quest to reclaim the dwarves' home and treasure from the dragon Smaug.
